# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SWAPI03/flyrank-ai-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Contract, in plain words (answers 1-3 of 5).**

1. **What one row means.** In the raw warehouse fact `fact_content_daily_performance`, one row =
   **one content page, on one day, for one client** (`report_date` x `client_hash_id` x
   `content_hash_id`). My lane (CTR / engagement opportunity scoring) aggregates those daily rows
   up to **one row per content page** over a chosen feature window — that aggregated page is my
   real unit of analysis.
2. **Which table(s).** `fact_content_daily_performance` (daily GSC/GA4 signals) as the spine, joined
   to `dim_content` for page metadata, with `dim_clients` for per-client history coverage. The
   query-mix table `fact_content_query_90d` is an optional later add.
3. **Which time window.** I develop on a **mid-panel month, `month=2026-03`**, as the feature
   window, and I keep the final month (June 2026, the `_sample` table) sealed as a test month — the
   last month is the natural outcome window of any past->future label, so developing on it would peek
   at the future. For the leak demo below I use March as the feature window and April as a short
   future outcome window.

In [9]:
# ---- Setup: connect DuckDB to the gated release using your Colab HF_TOKEN secret ----
%pip -q install duckdb huggingface_hub
import os, getpass, duckdb, numpy as np, pandas as pd

# Token order: env var -> Colab Secret (HF_TOKEN) -> prompt. Never paste a token into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL   = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"                                                   # mid-panel feature month
MAR   = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Confirm the connection: this counts Parquet metadata, not data, so it returns in seconds.
n = con.sql(f"SELECT COUNT(*) FROM {MAR}").fetchone()[0]
print(f"Connected. Rows in month={MONTH} partition: {n:,}")

Connected. Rows in month=2026-03 partition: 9,841,378


## 2. Fields: feature / label / context / excluded

**Contract, in plain words (answers 4-5 of 5).**

4. **What I predict or rank (label / proxy).** I rank visible pages by how far they **under-capture
   clicks for their search position**. The starter proxy is a position-adjusted CTR gap; the honest
   future label used in the leak demo below is *observed decline* — did a page's impressions fall by
   more than 20% from March to April.
5. **One thing I deliberately exclude.** Any **label-window column used as a feature** — above all
   `imp_april` (the April impressions the decline label is built from). Including it is textbook
   leakage, which I prove and then remove in section 3. I also exclude product decision flags, which
   the release does not ship anyway.

Field buckets for this lane:

| Bucket | Fields |
|---|---|
| **Features** (knowable at the decision moment) | March-window aggregates: `imp_march`, `clk_march`, `pos_march` (avg GSC position), `active_days_march`, `ctr_march` |
| **Label / proxy** | `is_declining` = `imp_april < 0.8 * imp_march` (observed future outcome) |
| **Context** (join / grouping only, never a feature) | `client_hash_id`, `content_hash_id`, `ga4_data_available` |
| **Excluded** (leakage or unsafe) | `imp_april` as a feature (label window); any raw URL / query / title — not in the release |

In [10]:
# Confirm the fields I named are really in the table (schema introspection, no heavy scan).
schema = con.sql(f"DESCRIBE SELECT * FROM {MAR} LIMIT 0").df()
print("Columns available in fact_content_daily_performance:")
print(schema[["column_name", "column_type"]].to_string(index=False))

Columns available in fact_content_daily_performance:
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemi

## 3. Verify it with queries (grain, counts, availability) + five features + the trap

Every contract claim gets a query. Below: **exactly three verification queries** on `month=2026-03`,
then a **five-feature frame**, then the **deliberate-leak experiment**.

### Three verification queries

**Q1 — grain:** is one row really `report_date` x `client` x `content`? (duplicates should be 0)
**Q2 — counts + date span:** how big is my March slice, and does it span the whole month?
**Q3 — availability with `IS TRUE`:** how many rows actually carry GA4 signal? The flag is
three-valued (TRUE / FALSE / NULL), so I filter with `IS TRUE`, never `= TRUE`.

In [11]:
# Q1 - GRAIN: prove (report_date, client_hash_id, content_hash_id) is unique in the month.
q1 = con.sql(f"""
    WITH t AS (SELECT report_date, client_hash_id, content_hash_id FROM {MAR})
    SELECT
      (SELECT COUNT(*) FROM t) AS total_rows,
      (SELECT COUNT(*) FROM (SELECT DISTINCT report_date, client_hash_id, content_hash_id FROM t)) AS distinct_keys
""").df()
dup = int(q1["total_rows"][0] - q1["distinct_keys"][0])
print(q1.to_string(index=False))
print(f"duplicate keys: {dup}   ->  grain confirmed" if dup == 0 else f"duplicate keys: {dup} (grain NOT clean)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  distinct_keys
    9841378        9841378
duplicate keys: 0   ->  grain confirmed


In [12]:
# Q2 - COUNTS + DATE SPAN: size of my slice and the dates it covers.
q2 = con.sql(f"""
    SELECT COUNT(*)                    AS rows,
           COUNT(DISTINCT content_hash_id) AS pages,
           COUNT(DISTINCT client_hash_id)  AS clients,
           MIN(report_date)            AS first_day,
           MAX(report_date)            AS last_day
    FROM {MAR}
""").df()
print(q2.to_string(index=False))

   rows  pages  clients  first_day   last_day
9841378 331437       55 2026-03-01 2026-03-31


In [13]:
# Q3 - AVAILABILITY with IS TRUE (three-valued flag: TRUE / FALSE / NULL).
q3 = con.sql(f"""
    SELECT COUNT(*)                                          AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)     AS ga4_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_true
    FROM {MAR}
""").df()
surv = q3["ga4_true"][0] / q3["total_rows"][0] * 100
print(q3.to_string(index=False))
print(f"GA4 signal present on {surv:.1f}% of March rows (IS TRUE). The rest are GSC-only or NULL --")
print("treating their zeros as 'no engagement' would be a mistake, which is why we filter IS TRUE.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  ga4_true  ga4_not_true
    9841378    413966       9427412
GA4 signal present on 4.2% of March rows (IS TRUE). The rest are GSC-only or NULL --
treating their zeros as 'no engagement' would be a mistake, which is why we filter IS TRUE.


### Five features (max), each knowable at the decision moment

All five are aggregated from the **March window only**, so each is knowable at the decision moment
(the end of March), before any future outcome exists:

- `imp_march` — sum of GSC impressions in March. *Knowable because it only sums in-window days.*
- `clk_march` — sum of GSC clicks in March. *Knowable because clicks are counted only within March.*
- `pos_march` — average GSC position in March (positions > 0). *Knowable because it averages ranking observed during March.*
- `active_days_march` — distinct March days with impressions. *Knowable because it counts exposure days inside the window.*
- `ctr_march` — 100 x clk_march / imp_march. *Knowable because it is derived only from in-window March clicks and impressions.*

I also pull `imp_april` **only to build the label** (April impressions) — it is not a feature.

In [14]:
# Build the five-feature frame from March, plus imp_april for the label (read only 2 months).
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
      SUM(CASE WHEN report_date <  DATE '2026-04-01' THEN gsc_impressions ELSE 0 END) AS imp_march,
      SUM(CASE WHEN report_date <  DATE '2026-04-01' THEN gsc_clicks      ELSE 0 END) AS clk_march,
      AVG(CASE WHEN report_date <  DATE '2026-04-01' AND gsc_avg_position > 0
               THEN gsc_avg_position END)                                            AS pos_march,
      COUNT(DISTINCT CASE WHEN report_date < DATE '2026-04-01' AND gsc_impressions > 0
                          THEN report_date END)                                      AS active_days_march,
      SUM(CASE WHEN report_date >= DATE '2026-04-01' THEN gsc_impressions ELSE 0 END) AS imp_april
    FROM read_parquet([
        '{REL}/fact_content_daily_performance/month=2026-03/*.parquet',
        '{REL}/fact_content_daily_performance/month=2026-04/*.parquet'
    ])
    GROUP BY 1, 2
    HAVING imp_march >= 100          -- enough March exposure to be worth ranking
""".format(REL=REL)).df()

feat["ctr_march"] = 100 * feat["clk_march"] / feat["imp_march"]
print(f"feature frame: {len(feat):,} content pages (one row = one page)")
print(feat[["content_hash_id", "imp_march", "clk_march", "pos_march",
            "active_days_march", "ctr_march"]].head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature frame: 101,441 content pages (one row = one page)
            content_hash_id  imp_march  clk_march  pos_march  \
0  content_7a105f548d9c6916     6523.0        7.0   7.209549   
1  content_a3ea9792f793ec72      453.0        0.0   3.307255   
2  content_36c36abc7650d7af     5630.0        6.0   6.724039   
3  content_a7da352b73b02668     4944.0       13.0   7.244844   
4  content_1855a661b4d36130      429.0        1.0   4.499519   

   active_days_march  ctr_march  
0                 31   0.107313  
1                 31   0.000000  
2                 31   0.106572  
3                 31   0.262945  
4                 31   0.233100  


### The trap: one label-derived column, on real warehouse data

The honest features are the five March signals. The label is *observed* April decline. Watch a quick
score with the honest features, then **add `imp_april`** — the very column the label is built from —
and watch it jump toward perfect. That jump is leakage, not skill. Then I delete the column and keep
the honest number. This is the notebook-02 lesson, performed on the warehouse.

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

feat["is_declining"] = (feat["imp_april"] < 0.8 * feat["imp_march"]).astype(int)  # observed future label
honest = ["imp_march", "clk_march", "pos_march", "active_days_march", "ctr_march"]
d = feat.dropna(subset=honest).copy()

base_rate = max(d["is_declining"].mean(), 1 - d["is_declining"].mean())
print(f"base rate (always predict majority): {base_rate:.3f}")

def quick_acc(cols):
    Xtr, Xte, ytr, yte = train_test_split(d[cols], d["is_declining"],
                                          test_size=0.25, random_state=42, stratify=d["is_declining"])
    m = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xtr, ytr)
    return accuracy_score(yte, m.predict(Xte))

honest_acc = quick_acc(honest)
leaky_acc  = quick_acc(honest + ["imp_april"])   # imp_april is the label window -> leakage
print(f"honest features  accuracy: {honest_acc:.3f}")
print(f"+ imp_april (LEAK) accuracy: {leaky_acc:.3f}   <- jumps toward perfect, and it is fake")

# Delete the leaked column and keep the honest number.
feat.drop(columns=["imp_april"], inplace=True, errors="ignore")
print(f"\nDeleted imp_april. Honest number I keep: {honest_acc:.3f} (vs base rate {base_rate:.3f}).")
print("Observed / directional: a leak-derived column looks amazing and teaches nothing; the honest")
print("gap over the base rate is the only figure worth reporting.")

base rate (always predict majority): 0.517
honest features  accuracy: 0.618
+ imp_april (LEAK) accuracy: 0.745   <- jumps toward perfect, and it is fake

Deleted imp_april. Honest number I keep: 0.618 (vs base rate 0.517).
Observed / directional: a leak-derived column looks amazing and teaches nothing; the honest
gap over the base rate is the only figure worth reporting.


## 4. Data limits

**One named limitation of my slice:** a March->April comparison is a **single adjacent-month
window**, so it cannot separate a real decline from **seasonality or consolidation** — a page can
drop in April because demand is seasonal, or because a sibling page absorbed its traffic, not because
the page got worse. The honest label needs several windows and group checks before it can carry
weight.

Two structural limits the data also imposes: the panel is **unbalanced** (clients start tracking at
different dates, so a fixed calendar window means different history depth per client — always check
`dim_clients.gsc_data_start`), and early rows are **GSC-only** with `ga4_data_available` FALSE or
NULL, so GA4 zeros are "not measured", not "no engagement". None of this data can prove a refresh
*caused* a recovery — this stays decision-support, not causal proof.

In [16]:
# Evidence for the unbalanced-panel limitation: history depth differs a lot across clients.
lim = con.sql(f"""
    SELECT COUNT(*)                                                    AS clients,
           COUNT(*) FILTER (WHERE gsc_data_start <= DATE '2025-06-30') AS clients_12mo_plus,
           MIN(gsc_data_start)                                         AS earliest_start,
           MAX(gsc_data_start)                                         AS latest_start
    FROM read_parquet('{REL}/dim_clients.parquet')
""".format(REL=REL)).df()
print(lim.to_string(index=False))
print("Different start dates => a single calendar window covers very different history per client.")

 clients  clients_12mo_plus earliest_start latest_start
     104                  9     2025-01-27   2026-06-02
Different start dates => a single calendar window covers very different history per client.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.